# Note

Apologies for the late homework.  This homework has 3 (multi-part) problems and is somewhat shorter than it looks.

Problem 1 foreshadows something which we'll look at on Tuesday of next week, the loop-erased random walk, which is crucial for quickly sampling a random spanning tree.  

Problems 2 and 3 were discussed in Thursday's in-class programming exercise.  Not so many people attended on Thursday's class - enough so that I felt it was important to include a brief summary of the lecture.  If you missed the class, please work with someone who was there!



# (1) Loop erased random walk

Let $G = (V, E)$ be a graph with no loops, no multiple edges.  A *simple random walk* in $G$ is a walk $b_0, b_1, b_2 \dots$ with $b_i \in V$ such that $b_i$ is chosen uniformly at random from the neighbors of $b_{i-1}$, for all $i \geq 1$.

One way to define a [loop erased random walk](https://en.wikipedia.org/wiki/Loop-erased_random_walk) is as follows:
 - perform a simple random walk in $G$ with vertices $b_0, b_1, \dots, b_k \in S$, stopping at the first time we step to a vertex $b_k$ in $S$.
  - As we do, we partially define a function $f:V \rightarrow V$: let $f(b_i) = b_{i+1}$.  If we've previously defined $f(b)$ for some vertex $b$ that we visit more than once, then overwrite $f(b)$ with the new value.

Once this is done: set $a_0 = b_0$, $a_{i+1} = f(a_{i})$ for $i \geq 1$, stopping at $a_j \in S$ for some $j \leq k$.  

(a) Explain the name "loop-erased random walk".  That is, explain why this process is the same process as the one on the wikipedia.

(b) Implement loop erased random walk in the $n \times n$ square grid, starting in (roughly) the center, and ending at one of the leftmost boundary edges in the grid (the next question will ask you to use a different ending set).  I strongly suggest you use the description above, as it is far faster than all the loop-erasing you'd otherwise be doing.  Store values of $f$ in a python dictionary.


# (2) Plain Changes - review

Reference: The Art of Computer Programming, by D.E. Knuth, Vol. 4, Ch. 7.2, Algorithm P.

Here is a description of *plain changes*, which I discussed in Thursday's class.  "Plain changes" is the name for a particular Hamiltonian cycle in the Cayley graph of $S_n$ - i.e. a procedure for iterating through all $n!$ permutations of $n$, one at a time, where each permutation differs from the next by a swap of adjacent elements.  

The name comes from "change ringing", which is a 17th century British tradition in which the $n$ bells in a large church are rung in all possible orders (one of the few musical things you can do with these heavy bells, which want more than anything else to ring out very loudly every two seconds; it takes all a person's strength and body weight pulling on a rope to make them do anything else).

Here is a recursive description of plain changes.  

Plain changes in $S_2$ is as follows:  

```
12, 21 
```

I checked in class (and you can check, after having read this question) that Plain Changes for $S_3$ is the following sequence of permutations: 

```
123   132   312   321   231   213
```

Now, suppose $n \geq 3$.  First, do plain changes for $S_{n-1}$, giving a sequence of permutations $id = \pi_1, \dots, \pi_{(n-1)!}$ such as the one above.  

Then, insert $n$ into these permutations in all ways, proceeding from right to left.

```
1234   1324   3124   3214   2314   2134
1243   1342   3142   3241   2341   2143
1423   1432   3412   3421   2431   2413
4123   4132   4312   4321   4231   4213
```

Finally, read these permutations as follows: read the first column top-to-bottom, the next column bottom-to-top, etc.  So,

```
1234 1243 1423 4123 4132 1432 1342 1324 3124 3142 ...  ... 2143 2134.
```

# Plain changes - Problem

Given a permutation $a \in S_n$ in one-line notation: define its *inversion table* $c_1, \dots, c_{n}$ as follows: $c_j$ is the number of elements less than $j$ which appear to the right of $j$ in $a$.  

(a) Write code that converts permutations to inversion tables.  Then: Let $T_n$ be the set of inversion tables.  Describe $T_n$.  Show that the map $S_n \rightarrow T_n$ which sends a permutation to its inversion table is a bijection.  Write code that converts inversion tables *back* to permutations, and test it.

(b) Compute the inversion tables for plain changes in $S_4$ and in $S_5$.  What do you notice?  Prove it. 

(c) Write a *non-recursive* implementation of plain changes.  There are breathtakingly efficient ones, but I don't need one of these.  Just write a piece of code which does not include a recursive function call, but which does generate the same sequence of permutations.  One approach is to use part (b); there are others.

Here is a recursive implementation of plain changes, for your reference.

In [3]:
from itertools import cycle

def plain_changes(n):
    if n ==2:
        yield (1,2)
        yield (2,1)
    else:
        column_is_even = cycle([True, False])
        for tau, read_downward in zip(plain_changes(n-1), column_is_even):
            if read_downward:
                column_indices = reversed(range(n))
            else:
                column_indices = range(n)
            for i in column_indices:
                yield tau[:i] + tuple([n]) + tau[i:]

list(plain_changes(4))
                

[(1, 2, 3, 4),
 (1, 2, 4, 3),
 (1, 4, 2, 3),
 (4, 1, 2, 3),
 (4, 1, 3, 2),
 (1, 4, 3, 2),
 (1, 3, 4, 2),
 (1, 3, 2, 4),
 (3, 1, 2, 4),
 (3, 1, 4, 2),
 (3, 4, 1, 2),
 (4, 3, 1, 2),
 (4, 3, 2, 1),
 (3, 4, 2, 1),
 (3, 2, 4, 1),
 (3, 2, 1, 4),
 (2, 3, 1, 4),
 (2, 3, 4, 1),
 (2, 4, 3, 1),
 (4, 2, 3, 1),
 (4, 2, 1, 3),
 (2, 4, 1, 3),
 (2, 1, 4, 3),
 (2, 1, 3, 4)]

# (3) Toposort on Young Tableaux - review

Reference: The Art of Computer Programming by D.E. Knuth, Vol. 4A, Chapter 7.2, Algorithm V.

Let $P$ be a partially ordered set.  The $n$-chain $C_n$ is the set $\{1,2,\dots, n\}$ with the usual (total) order on $\mathbb{N}$.  A *linear extension of $P$* is a map from $P$ to $C_n$ which preserves the poset structure.  

Let $L_0$ be a fixed linear extension of $P$, called the *reference extension*.

For this problem:  $P$ is going to be the Young diagram of a partition $\lambda$; its linear extensions are going to be the standard Young tableaux of shape $\lambda$, and the *reference tableau* $L_0$ is going to be the one which writes the numbers 1 through $n$ in increasing order, top-to-bottom, then left-to-right.  For instance, when $\lambda = 431$,  $L_0$ is the tableau
```
1468
257
3
```
We regard $L_0$ as the ''names'' of the elements in the poset $P$.

As we did in class on Thursday: given another linear extension $L$, define a permutation $\pi$ as follows: for each $i$, find the cell in $L$ which contains the number $i$; then $\pi(i)$ is the contents of the corresponding cell in $L_0$.

We say $a \stackrel{\pi}{\leq} b$  if $a$ appears to the left of $\pi$ in b.

we say $a \stackrel{\lambda}{\leq} b$ if $a$ appears weakly northwest of $b$ in $\lambda$.  

If $\pi$ is a permutation of $n$ such that $a \stackrel{\pi}{\leq} b$ whenever $a \stackrel{\lambda}{\leq} b$, we say that $\pi$ is a *topological sort* or *toposort* of $P$.

# Toposort on Young Tableaux - Problem

(a) Explain why standard tableaux of shape $\lambda$ are in bijection with toposorts of $P$.

(b) Finish thursday's in-class exercise, which finds all toposorts of $P$.  The algorithm is recursive: 
- remove $n$ from $P$ to get a smaller poset $\hat{P}$.
- Generate all topological sorts of $\hat{P}$.  Each is a permutation $a_1 \dots a_{n-1} = \pi \in S_{n-1}$ in one-line notation.
- For each: insert $n$ in the last position, the second-last position, etc, producing a sequence of permutations:

\begin{align*}
a_1\dots a_{n-1}n, \\
a_1\dots a_{n-2}n a_{n-1}, \\
a_1 \dots a_{n-3} n a_{n-2} a_{n-1}, 
\end{align*}

Stop at the first line $k$ for which $a_{n-k} \stackrel{\lambda}{<} n$.  Before this point, all the generated permutations are toposorts; afterwards, none of them are.  Explain why this works.

(c) Test your algorithm.  Generate the five topological sorts of $P$ when $\lambda$ is the partition $(2,2,1)$, display them as Young tableaux. Then, generate all topological sorts of $P$ when $P$ is the partition $(3,3,3)$; there should be 48 of them.

Note: I was going to ask for a "non-recursive toposort" - but I didn't want to make this assignment too long.  If you need to implement this yourself at some point, I suggest reading the Knuth reference, which has an astonishingly simple implementation.
